## 1. Motivation and model overview

The **ball-drive risk** component estimates the probability that the current ball carrier loses possession during a canonical one-second ball drive. It is the action-specific risk $d_t$ used by PRESS to construct the possession-loss hazard. The event concerns loss of the current player's control during the drive, not a failed pass.

The model combines two complementary research traditions. Bekkers' pressing-intensity formulation motivates estimating how quickly defenders can reach a moving ball carrier [@bekkers2025]. Spearman's pitch-control framework motivates representing arrival uncertainty and the rate at which a player can establish control [@spearman2017; @spearman2018beyond]. PRESS adapts these ideas to a player **already in possession**: a defender must be able to reach the carrier and, once reachable, can generate a successful defensive intervention at a certain *tackle rate*. This rate plays a role analogous to the control rate in pitch control, but measures the intensity of possession loss. PRESS does not reproduce Bekkers' aggregate pressing score.

Each defender contributes a time-varying cause-specific hazard. A shared survival process combines these contributions into the probability of retaining the ball over one second. The one-second action convention follows the modular EPV perspective of Fernández, Bornn and Cervone [@fernandez2019; @fernandez2021].

The approach is **physics-informed**: motion and reachability follow an explicit physical approximation, whereas tackle intensity is estimated from observed outcomes. We first introduce a simple log-linear, proportional-hazards-inspired rate and then extend it with penalized additive spline effects. The final fitted spline model is used for inference.

## 2. Motion and defender interception time

### Projected ball-carrier trajectory

Let $x_B(0)$ and $v_B(0)$ denote the carrier's observed position and velocity at the evaluation frame. During the hypothetical drive, the carrier follows a constant-velocity projection:

$$
x_B(t)=x_B(0)+v_B(0)t,\qquad 0\le t\le 1.
$$

At each integration time, defender $i$ targets the projected carrier position $x_B(t)$. The carrier is represented as a 2d point. Let $d_i(t)$ denote the defender's distance to that target, and let $u_i=\lVert v_i(0)\rVert$ be the defender's observed initial speed.

### Travel and turning time

For initial speed $u_i\le v_{\max}$, the motion approximation uses linear acceleration $a=a_{\mathrm{linear}}$ up to an effective speed $v_{\max}$. Define

$$
t_a=\max\left\{\frac{v_{\max}-u_i}{a},0\right\},
\qquad
d_a=u_it_a+\frac12 at_a^2.
$$

The estimated travel time to the projected target is

$$
t_{\mathrm{move},i}(t)=
\begin{cases}
\dfrac{\sqrt{u_i^2+2ad_i(t)}-u_i}{a}, & d_i(t)\le d_a,\\[6pt]
t_a+\dfrac{d_i(t)-d_a}{v_{\max}}, & d_i(t)>d_a.
\end{cases}
$$

If the observed initial speed exceeds $v_{\max}$, the model retains that initial speed without additional acceleration and uses $t_{\mathrm{move},i}(t)=d_i(t)/u_i$. Thus $v_{\max}$ is not a hard cap on observed initial speeds.

Following Pleuler's use of a heading-dependent turning penalty [@pleuler2020handbook], PRESS parameterizes the defender's reorientation with a simplified turning delay. With turning angle $\theta_i(t)$ in radians and turning-acceleration parameter $a_{\mathrm{turn}}$,

$$
t_{\mathrm{turn},i}(t)=\frac{u_i\theta_i(t)}{a_{\mathrm{turn}}}.
$$

This expression follows from a turning radius $R=u_i^2/a_{\mathrm{turn}}$ and arc time $R\theta_i/u_i$. The implementation adds a delay but does not propagate the defender along a turning arc. The arrival-time estimate is

$$
\tau_i(t)=t_{\mathrm{move},i}(t)+t_{\mathrm{turn},i}(t).
$$

For the existing near-target convention, $\tau_i(t)=0$ whenever $d_i(t)\le0.1$ m. This does not imply certain reachability.

## 3. Probabilistic defender reachability

As in the pitch-control approach, deterministic arrival times are converted into a smooth reachability gate. With logistic arrival-time dispersion $s$ (in seconds), defender $i$ has reachability

$$
F_i(t)=\left[1+\exp\left\{-\frac{\pi\,[t-\tau_i(t)]}{\sqrt{3}\,s}\right\}\right]^{-1}.
$$

The parameter $s$ is the standard deviation of the corresponding logistic arrival-time uncertainty. The gate increases when the defender can reach the projected carrier position sooner relative to elapsed time.

## 4. Tackle intensity

### From a log-linear rate to additive splines

The second ingredient is the instantaneous rate at which a reachable defender can cause loss of control. Inspired by the control-rate mechanism of pitch control, PRESS assigns defender $i$ a nonnegative tackle rate $\lambda_i(t)$, with units s$^{-1}$. Unlike a probability, a rate may exceed one.

A simple starting point is a log-linear, proportional-hazards-inspired model:

$$
\lambda_i(t)=\lambda_0\exp\!\left\{\beta_v\lVert v_B\rVert+\beta_B B(t)\right\}.
$$

Here $\lambda_0$ is a baseline rate, $\lVert v_B\rVert$ is carrier speed, and $B(t)$ is signed minimum distance to the pitch boundaries. This model assumes linear covariate effects on the log-rate. The final PRESS specification relaxes that restriction through penalized additive spline effects and adds a defender-specific approach angle:

$$
\boxed{\log\lambda_i(t)=\alpha+g_v(\lVert v_B\rVert)+g_B(B(t))+g_\phi(\phi_i(t)).}
$$

Here $\alpha=\log\lambda_0$ is the log baseline tackle-rate parameter. The fitted specification uses linear terms plus restricted cubic nonlinear terms for speed and boundary distance; angle has a standardized linear term and two nonlinear terms. There are no interaction terms. The model retains a multiplicative rate structure, but its full cause-specific hazards are time-varying through reachability and the evaluated covariates. It should not be interpreted as a textbook Cox model with constant hazard ratios.

### Why these covariates?

**Carrier speed.** Schepers et al. identify movement speed as an important predictor of dribbling outcomes [@schepers2025dribble]. Greater speed can make fine ball control more difficult, increasing vulnerability to intervention. The fitted PRESS model supports an increase in tackle intensity at higher carrier speeds. The spline allows the magnitude and shape of that effect to be estimated without imposing a strictly log-linear response.

**Boundary proximity.** Available space is a central aspect of defensive pressure, including SkillCorner's emphasis on spatial constraints [@skillcorner2026]. Near a boundary, the carrier has fewer possible directions in which to evade an intervention. PRESS uses the signed minimum distance to **all four** pitch boundaries, recomputed along the projected path:

$$
B(t)=\min\!\left\{\frac L2-x_B(t),\frac L2+x_B(t),\frac W2-y_B(t),\frac W2+y_B(t)\right\}.
$$

This differs from a sideline-only distance and can become negative beyond the pitch boundary.

**Defender approach angle.** Spatial constraints also depend on whether a defender approaches from ahead, behind, or the side. Motivated by the role of the carrier's available field of view and escape directions in pressure interpretation, PRESS measures $\phi_i(t)$ as the angle between carrier velocity and the defender's predicted final approach direction towards $x_B(t)$. It is a geometric proxy, not observed body orientation or gaze. The angle contribution is masked when carrier speed is below 1 m/s or the direction is undefined.

These three covariates describe complementary mechanisms: control difficulty at speed, geometric restrictions near the pitch boundary, and the direction of defensive engagement. The spline terms quantify their associations in the fitted model.

## 5. From defender hazards to possession-loss probability

Defender $i$ contributes the effective cause-specific hazard

$$
h_i(t)=F_i(t)\lambda_i(t).
$$

Reachability gates the tackle intensity: even a high underlying tackle rate contributes little when the defender is unlikely to reach the carrier. Under the model's competing-risk construction, all defenders share a single survival probability,

$$
S(t)=\exp\!\left\{-\int_0^t\sum_i h_i(u)\,du\right\}.
$$

The probability that defender $i$ is the cause of loss by time $t$ is

$$
P_i(t)=\int_0^t S(u)h_i(u)\,du.
$$

Therefore,

$$
\boxed{P(\mathrm{loss\ by}\ t\mid\mathrm{drive})=1-S(t)=\sum_i P_i(t).}
$$

The **one-second ball-drive risk** passed to the wider PRESS framework is $d_t=1-S(1)$, where the evaluation state is understood to be fixed at the start of the hypothetical action. Multiple defenders affect risk through their individual reachability and rates; coordinated defensive behaviour is not explicitly modelled.

## 6. Motion parameters

The model uses $v_{\max}=7$ m/s and $a_{\mathrm{linear}}=a_{\mathrm{turn}}=7$ m/s$^2$. These are effective common movement capabilities, motivated by published models and checked against the tracking distribution.

| Parameter | Literature rationale | Tracking consistency check |
|---|---|---|
| Defender speed $v_{\max}=7$ m/s | Approximately the 25.1 km/h sprint threshold in Bradley et al. [@bradley2009running]. | Defensive-engagement event frames with speed $>2$ m/s: Q95 = **7.050 m/s**. |
| Linear acceleration $a_{\mathrm{linear}}=7$ m/s$^2$ | The acceleration used by Spearman's movement model [@spearman2018beyond]. | Engagement acceleration peaks above 3 m/s$^2$: Q99 = **6.966 m/s$^2$**. |
| Turning acceleration $a_{\mathrm{turn}}=7$ m/s$^2$ | A heading-dependent delay follows the idea in Pleuler [@pleuler2020handbook]; PRESS sets its scale equal to the linear-acceleration parameter. | Equality with linear acceleration is a modelling convention; it is not separately identified by these quantiles. |

The speed filter removes stationary/walking observations for this descriptive check. These conditional quantiles are consistency checks for rounded model scales, not physiological ceilings or independently estimated caps. Event-frame observations and event peaks use different sampling units. The [speed audit](../../data/examples/ball_drive/defender_speed_check.csv), [engagement quantiles](../../data/benchmarks/ball_drive/engagement_quantiles.csv) and [kinematic conventions](../../docs/kinematic-parameter-conventions.md) provide the definitions and sources.

Positions and velocities enter the model directly; observed acceleration is not an input. The current velocities use the offline centred seven-frame tracking estimate.


## 7. Likelihood, priors and posterior estimates {#likelihood-priors-and-posterior-estimates}

### Likelihood: the outcome of a one-second carry

For carry $j$, let $Y_j=1$ denote a loss and $Y_j=0$ successful retention over the canonical horizon. Given the observed initial state and parameters $\theta$, the competing-risk model supplies

$$d_j(\theta)=1-S_j(1;\theta).$$

The outcome likelihood is Bernoulli:

$$\boxed{\mathcal L(\theta;D)=\prod_j
 d_j(\theta)^{Y_j}\,[1-d_j(\theta)]^{1-Y_j}.}$$

Equivalently,

$$\log\mathcal L=\sum_j\left\{Y_j\log[1-S_j(1)]+(1-Y_j)\log S_j(1)\right\}.$$

A loss before one second contributes $1-S_j(1)$: its observed failure time does not shorten the forecast horizon. A successful full second contributes $S_j(1)$. The fit learns which initial configurations make the one-second outcome more or less likely; it does not fit the exact tackle time or the observed tackler's identity.

### Priors: positive scales and regularized covariate effects

Write $\alpha=\log\lambda_0$ and $\ell=\log s$. The fitted model uses independent priors

$$\alpha\sim\mathcal N(0,1),\qquad
\beta_v,\beta_B\sim\mathcal N(0,1),$$

$$\boxed{s\sim\operatorname{Gamma}(\text{shape}=2.5,\ \text{rate}=5).}$$

Thus $\lambda_0$ has a lognormal prior with log-location 0 and log-SD 1. The Gamma prior keeps $s$ positive, has mean **0.5 s** and SD **0.316 s**, and places it on the same broad time scale as the arrival dispersion in Spearman's models [@spearman2017; @spearman2018beyond]. These exact prior choices belong to PRESS; they are not published estimates copied from those papers.

Spline coefficients have independent zero-mean Gaussian priors. Their precision increases with the curvature penalty, shrinking unsupported bends toward simpler effects. The final specification uses penalty multipliers **1** for the speed/boundary nonlinear terms and **100** for the angle nonlinear terms; the standardized linear angle term has a unit-variance Normal prior. How these strengths were selected is documented in the [benchmark](../../benchmarks/01_ball_drive_risk.qmd).

The posterior combines the one-second likelihood with these priors:

$$p(\theta\mid D)\propto\mathcal L(\theta;D)\,p(\theta).$$

Optimization uses the log coordinates. Consequently, the prior density for $\ell$ includes the transformation Jacobian:

$$p_\ell(\ell)=p_s(e^\ell)e^\ell
\ \propto\ \exp\{2.5\ell-5e^\ell\}.$$

If $\gamma$ collects the spline/angle coefficients and $Q$ their Gaussian prior precision, the minimized negative log posterior, up to constants, is

$$-\log\mathcal L
+\tfrac12(\alpha^2+\beta_v^2+\beta_B^2)
+5e^\ell-2.5\ell+\tfrac12\gamma^TQ\gamma.$$

### Joint MAP and approximate marginal posteriors

The full fitted parameter vector is estimated jointly by maximum a posteriori (MAP). A Laplace approximation uses the local curvature of that joint posterior:

$$\theta\mid D\ \dot\sim\ \mathcal N(\hat\theta,\Sigma),\qquad
\Sigma=\left[\nabla^2\{-\log p(\theta\mid D)\}_{\hat\theta}\right]^{-1}.$$

The covariance includes the tackle coefficients and their dependence on the two positive scales. Its marginal log-coordinate approximations are

$$\log\lambda_0\mid D\ \dot\sim\ \mathcal N(-1.065479,\,0.014572626),$$
$$\log s\mid D\ \dot\sim\ \mathcal N(-1.205201,\,0.003378886).$$

The second argument here is a **variance**. Transforming these marginals gives approximate lognormal posteriors:

| Parameter | Transformed joint MAP | Approximate posterior mean | 95% credible interval |
|---|---:|---:|---:|
| Arrival dispersion $s$ | **0.299632 s** | 0.300139 s | **0.267368–0.335789 s** |
| Baseline tackle rate $\lambda_0$ | **0.344563 s$^{-1}$** | 0.347082 s$^{-1}$ | **0.271965–0.436539 s$^{-1}$** |

For either parameter, the interval is $\exp\{\hat m\pm1.96\sqrt{\Sigma_{kk}}\}$ and the approximate posterior mean is $\exp(\hat m+\Sigma_{kk}/2)$. A transformed joint MAP estimate is not the same as the marginal posterior mean. Package inference uses the joint MAP point parameters; it does not average predictions over posterior draws.

![Actual prior densities and approximate posterior marginals for the two fitted scales. Horizontal axes are logarithmic.](../../figures/vignette_01_drive_risk.png)

$\lambda_0$ is an intercept-scale parameter in the saved spline basis: it is not an average tackle probability or necessarily the rate at an ordinary observed state. $s$ expresses temporal arrival uncertainty. The intervals concern parameter uncertainty conditional on this model, the fixed movement assumptions and selected prior strengths. They do not include model-selection uncertainty, systematic annotation error or a correction for within-match dependence. [Posterior summaries and prior definitions](../../data/examples/ball_drive/posterior_summary.json).


In [5]:
from pathlib import Path
import json
import sys

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from scripts.build_ball_drive_vignette import build_parameter_figure

build_parameter_figure(root)
summary = json.loads((root / "data/examples/ball_drive/posterior_summary.json").read_text())
for name, item in summary["parameters"].items():
    print(name, "MAP:", item["transformed_joint_map"],
          "posterior mean:", item["mean"], "95% interval:", item["credible_interval_95"])


lambda_0 MAP: 0.34456260008941997 posterior mean: 0.3470823597826589 95% interval: [0.2719653191366462, 0.43653869455586813]
s MAP: 0.2996318904783872 posterior mean: 0.30013852930091706 95% interval: [0.26736788098345543, 0.33578928576393846]


The dispersion is model-specific. Spearman's pass model reports 0.45 s, and a later OBSO formulation reports a MAP estimate of 0.54 s under a different estimation setup [@spearman2017; @spearman2018beyond]. Differences in movement approximation, data preparation, cohort selection, and tackle-rate flexibility may affect the fitted $s$, but their separate effects have not been identified here. A smaller estimate does **not** establish better tracking quality or less physiological variability. The [pass-probability vignette](../02_pass_probability/02_pass_probability.ipynb) discusses its independently fitted arrival parameter.

## 8. Real-match example: Velupillay loses the ball to Payne {#real-match-example-velupillay-loses-the-ball-to-payne}

**Wellington Phoenix–Melbourne Victory, 12 April 2025, 37:30.7.** Nishan Velupillay is carrying close to the touchline under pressure from Tim Payne and Alex Rufer. Both are annotated as `on_ball_engagement` events of type `recovery_press`, associated with this individual possession and active at the selected initial frame.

The package's fitted model assigns **60.5%** probability of loss within the next second. Payne contributes **50.6 percentage points** and Rufer **9.9**; the other defenders contribute negligibly. Payne therefore accounts for about **83.7% of the total modeled risk**, not an 83.7% unconditional chance of winning the ball.

Velupillay's loss is recorded **0.3 s later**. Payne starts the next possession as a recovery another **0.2 s later**. The ball stays in play, with no recorded foul or interruption. The predicted principal threat and observed recovering player agree.

This case was selected for illustration from high-risk held-out losses. Its original matching-fold probability was **59.1%**; the plots below recompute **60.5%** using the bundled full-data model. The two predictions have different fitting scopes. This selected case is not an additional performance benchmark.

### Load the real match and evaluate the initial frame

The code uses processed SkillCorner tracking through the package's public match and frame APIs. Set `SKILLCORNER_OPENDATA` to your local open-data repository; the relative processed-match location is recorded with the case. [Data setup](../../data.qmd) · [Case provenance](../../data/examples/ball_drive/real_loss_case.json).


In [8]:
import os
import matplotlib.pyplot as plt
from PRESS import PRESS
from PRESS.model import BallDriveModel
from PRESS.visualization import plot_pi
from scripts.build_ball_drive_vignette import finish_case_plot

case = json.loads((root / "data/examples/ball_drive/real_loss_case.json").read_text())
opendata = Path(os.environ.get("SKILLCORNER_OPENDATA", root.parent / "opendata"))
match_dir = opendata / case["processed_match_relative"]
press = PRESS.from_skillcorner(match_dir)
frame_id = case["frame_id"]
initial_frame = press.get_frame(frame_id)
state = press.get_state(frame_id)
press.compute_ball_drive_loss(frame_id, model=BallDriveModel.pretrained())
result = press.get_ball_drive_result(frame_id)
print("Initial frame:", frame_id, "carrier:", state["player"])
print("Probability of loss within one second:", float(result["Ball drive loss probability"]))
print("Defender-specific loss contributions:",
      dict(zip(state["defender_ids"], map(float, result["Individual tackle probabilities"]))))

plot_pi(press, frame_id, figsize=(10, 6.8), title_fontsize=11, legend_fontsize=9)
finish_case_plot(press, case, "start", root)


Initial frame: 24807 carrier: away_17
Probability of loss within one second: 0.6048426942292121
Defender-specific loss contributions: {'home_1': 6.942254043870134e-17, 'home_14': 0.09850824650274907, 'home_15': 6.3215088506533556e-06, 'home_19': 4.055082811115639e-17, 'home_25': 7.390739519400412e-12, 'home_27': 3.302764419958879e-11, 'home_3': 6.775913042394245e-10, 'home_4': 2.0721844021190714e-12, 'home_6': 0.5063281254975288, 'home_7': 2.1654193295929396e-17, 'home_9': 1.6978263251630381e-15}


### Initial state: forecast and defender contributions

![Initial real tracking frame. Defender colors show their absolute cause-specific loss probabilities; their contributions sum to the total risk. The star marks Velupillay.](../../figures/vignette_01_drive_pitch.png)

The figure uses `PRESS.visualization.plot_pi`. Colors encode each defender's contribution to loss over one second, not a rate or a conditional share. It is a close-up of the observed frame; velocity arrows show tracking velocities on a common 0.25-second drawing scale.


In [10]:
from PRESS.visualization import plot_frame

end_frame = case["recovery_frame"]
observed_end = press.get_frame(end_frame)
print("Observed loss frame:", case["loss_frame"])
print("Next control:", observed_end.player_possession.iloc[0], "at frame", end_frame)
assert observed_end.player_possession.iloc[0] == "home_6"  # Tim Payne

plot_frame(press, end_frame, figsize=(10, 6.8), title_fontsize=11, legend_fontsize=9)
finish_case_plot(press, case, "end", root)


Observed loss frame: 24810
Next control: home_6 at frame 24812


### End state: Payne in possession

![Observed recovery 0.5 seconds after the initial frame, shown with the package's tracking-frame visualization and the same spatial window. The star now marks Payne.](../../figures/vignette_01_drive_end_pitch.png)

This is an observed later frame drawn with `PRESS.visualization.plot_frame`; the title repeats the initial forecast for reference. Future tracking and the recovering player's identity are used only to check and describe the outcome, not to compute the initial prediction.


## Scope and limitations

The model uses a point-carrier approximation, constant carrier velocity over the hypothetical drive, simplified defender turning, and a reachability-gated tackle process. It does not explicitly represent body posture, foot-level ball control, individual skill, or fully coordinated defensive behaviour. Schepers et al. [@schepers2025dribble] motivate investigating richer dribbling and posture features, but overlap between successful and unsuccessful examples in the current 2D covariates does not establish that all useful 2D information has been exhausted. Shorter-horizon probabilities follow from the survival formulation but have not been separately calibrated.

## References and next steps

Bekkers [@bekkers2025] motivates moving-target pressing and reachability. Spearman [@spearman2017; @spearman2018beyond] motivates arrival uncertainty and control rates. Fernández, Bornn and Cervone [@fernandez2019; @fernandez2021] motivate the canonical action decomposition. Stalpaert et al. [@stalpaert2022] provide a carry-model benchmark, and Schepers et al. [@schepers2025dribble] motivate the dribbling covariates. SkillCorner [@skillcorner2026] motivates the interpretation of spatial constraints.

The equations above describe the PRESS formulation rather than a reproduction of any single cited model. Cohort selection, held-out results and comparisons are documented in the [ball-drive benchmark](../../benchmarks/01_ball_drive_risk.qmd). The pass-success component is described in the [next model vignette](../02_pass_probability/02_pass_probability.ipynb).